# Improvements: Preprocessing + SAHI + Domain Augmentation

This notebook implements and compares **three improvements** on top of the YOLOv8 baseline:

1. **Underwater Image Enhancement** — CLAHE + gray world white balance before inference
2. **SAHI (Slicing Aided Hyper Inference)** — tiled inference for small trash fragments
3. **Domain Augmentation Training** — train with synthetic haze/color cast for better robustness

Each improvement is tested against the baseline and results are compared.

In [ ]:
# ── Cell 1: Setup ─────────────────────────────────────────────────────────
!pip install -q ultralytics sahi

In [ ]:
# ── Cell 2: Mount + paths ────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import sys, os
import cv2, numpy as np
import matplotlib.pyplot as plt

DRIVE_BASE   = Path('/content/drive/MyDrive/underwater_waste_detection')
YOLO_DS_DIR  = DRIVE_BASE / 'trashcan_yolo'
WEIGHTS_DIR  = DRIVE_BASE / 'weights'
RESULTS_DIR  = DRIVE_BASE / 'improvement_results'
RESULTS_DIR.mkdir(exist_ok=True)

REPO_DIR = Path('/content/underwater-waste-detection')
if not REPO_DIR.exists():
    !git clone https://github.com/omprxkash/underwater-waste-detection.git {REPO_DIR}
sys.path.insert(0, str(REPO_DIR / 'src'))
os.chdir(REPO_DIR)

BASELINE_WEIGHTS = str(WEIGHTS_DIR / 'yolov8n_best.pt')

## Improvement A: Underwater Image Enhancement

In [ ]:
# ── Cell 3: Enhancement functions ────────────────────────────────────────
from preprocess import enhance_underwater, apply_clahe, gray_world_white_balance

# Visualize enhancement effect on sample images
val_images = list((YOLO_DS_DIR / 'images' / 'val').glob('*.jpg'))
samples = val_images[:4]

fig, axes = plt.subplots(len(samples), 2, figsize=(12, 4*len(samples)))
for i, img_path in enumerate(samples):
    original = cv2.imread(str(img_path))
    enhanced = enhance_underwater(original)

    axes[i][0].imshow(cv2.cvtColor(original, cv2.COLOR_BGR2RGB))
    axes[i][0].set_title('Original', fontweight='bold')
    axes[i][0].axis('off')

    axes[i][1].imshow(cv2.cvtColor(enhanced, cv2.COLOR_BGR2RGB))
    axes[i][1].set_title('Enhanced (CLAHE + White Balance)', fontweight='bold')
    axes[i][1].axis('off')

plt.suptitle('Underwater Image Enhancement Effect', fontweight='bold', fontsize=13)
plt.tight_layout()
plt.savefig(str(RESULTS_DIR / 'enhancement_comparison.png'), dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Cell 4: Baseline mAP (no preprocessing) ───────────────────────────────
import yaml
from ultralytics import YOLO

yaml_path = str(REPO_DIR / 'configs' / 'dataset_colab.yaml')
# Regenerate if needed
dataset_config = {'path': str(YOLO_DS_DIR), 'train': 'images/train',
                  'val': 'images/val', 'test': 'images/test',
                  'nc': 3, 'names': {0: 'trash', 1: 'bio', 2: 'rov'}}
with open(yaml_path, 'w') as f:
    yaml.dump(dataset_config, f)

baseline_model = YOLO(BASELINE_WEIGHTS)
baseline_val = baseline_model.val(data=yaml_path, verbose=False)

print(f'Baseline mAP@0.5: {float(baseline_val.box.map50):.4f}')
print(f'Baseline mAP@0.5:0.95: {float(baseline_val.box.map):.4f}')

In [ ]:
# ── Cell 5: Train with enhancement-augmented dataset ─────────────────────
# Pre-process all training images with CLAHE + white balance, save to new dir
import shutil

ENHANCED_DS = DRIVE_BASE / 'trashcan_yolo_enhanced'

if not (ENHANCED_DS / 'images' / 'train').exists():
    print('Creating enhanced dataset...')
    for split in ['train', 'val', 'test']:
        src_imgs = YOLO_DS_DIR / 'images' / split
        src_lbls = YOLO_DS_DIR / 'labels' / split
        dst_imgs = ENHANCED_DS / 'images' / split
        dst_lbls = ENHANCED_DS / 'labels' / split
        dst_imgs.mkdir(parents=True, exist_ok=True)
        dst_lbls.mkdir(parents=True, exist_ok=True)

        # Enhance images
        for img_file in src_imgs.glob('*.jpg'):
            frame = cv2.imread(str(img_file))
            if frame is not None:
                enhanced_frame = enhance_underwater(frame)
                cv2.imwrite(str(dst_imgs / img_file.name), enhanced_frame)

        # Copy labels unchanged
        for lbl_file in src_lbls.glob('*.txt'):
            shutil.copy2(lbl_file, dst_lbls / lbl_file.name)

        n = len(list(dst_imgs.glob('*.jpg')))
        print(f'  {split}: {n} enhanced images')
else:
    print('Enhanced dataset already exists.')

# Write YAML for enhanced dataset
enhanced_yaml = str(REPO_DIR / 'configs' / 'dataset_enhanced.yaml')
enhanced_config = {'path': str(ENHANCED_DS), 'train': 'images/train',
                   'val': 'images/val', 'test': 'images/test',
                   'nc': 3, 'names': {0: 'trash', 1: 'bio', 2: 'rov'}}
with open(enhanced_yaml, 'w') as f:
    yaml.dump(enhanced_config, f)
print(f'Enhanced dataset YAML: {enhanced_yaml}')

In [ ]:
# ── Cell 6: Train on enhanced dataset ────────────────────────────────────
enhanced_model = YOLO('yolov8n.pt')
enhanced_model.train(
    data=enhanced_yaml,
    epochs=100,
    imgsz=640,
    batch=16,
    device=0,
    project=str(DRIVE_BASE / 'runs'),
    name='yolov8n_enhanced',
    optimizer='AdamW',
    lr0=0.001,
    warmup_epochs=3,
)
print('Enhanced model training complete.')

In [ ]:
# ── Cell 7: Compare baseline vs enhanced ─────────────────────────────────
import pandas as pd

enhanced_best = str(DRIVE_BASE / 'runs' / 'yolov8n_enhanced' / 'weights' / 'best.pt')
enhanced_model_eval = YOLO(enhanced_best)
enhanced_val = enhanced_model_eval.val(data=enhanced_yaml, verbose=False)

comparison = pd.DataFrame([
    {
        'Model': 'YOLOv8n (baseline)',
        'Preprocessing': 'None',
        'mAP@0.5': round(float(baseline_val.box.map50), 4),
        'mAP@0.5:0.95': round(float(baseline_val.box.map), 4),
        'Precision': round(float(baseline_val.box.mp), 4),
        'Recall': round(float(baseline_val.box.mr), 4),
    },
    {
        'Model': 'YOLOv8n + CLAHE + WB',
        'Preprocessing': 'CLAHE + Gray World WB',
        'mAP@0.5': round(float(enhanced_val.box.map50), 4),
        'mAP@0.5:0.95': round(float(enhanced_val.box.map), 4),
        'Precision': round(float(enhanced_val.box.mp), 4),
        'Recall': round(float(enhanced_val.box.mr), 4),
    },
])

print('\n' + '='*70)
print('IMPROVEMENT COMPARISON')
print('='*70)
print(comparison.to_string(index=False))
comparison.to_csv(str(RESULTS_DIR / 'preprocessing_comparison.csv'), index=False)

## Improvement B: SAHI — Small Object Detection

In [ ]:
# ── Cell 8: SAHI inference ────────────────────────────────────────────────
# SAHI slices the image into overlapping tiles, runs detection on each,
# then merges predictions with NMM. Significantly improves small object recall.

from sahi import AutoDetectionModel
from sahi.predict import get_sliced_prediction

sahi_model = AutoDetectionModel.from_pretrained(
    model_type='ultralytics',
    model_path=BASELINE_WEIGHTS,
    confidence_threshold=0.25,
    device='cuda',
)

# Regular inference vs SAHI on a sample image
sample_img = val_images[0]

# Standard inference
std_result = baseline_model(str(sample_img), conf=0.25, verbose=False)[0]
print(f'Standard inference: {len(std_result.boxes)} detections')

# SAHI sliced inference
sahi_result = get_sliced_prediction(
    str(sample_img),
    sahi_model,
    slice_height=320,
    slice_width=320,
    overlap_height_ratio=0.2,
    overlap_width_ratio=0.2,
)
print(f'SAHI inference: {len(sahi_result.object_prediction_list)} detections')

In [ ]:
# ── Cell 9: Visualize SAHI vs standard ───────────────────────────────────
frame = cv2.imread(str(sample_img))

# Draw standard detections
std_vis = frame.copy()
for box in std_result.boxes:
    x1,y1,x2,y2 = map(int, box.xyxy[0])
    cv2.rectangle(std_vis, (x1,y1),(x2,y2),(0,200,0),2)

# Draw SAHI detections
sahi_vis = frame.copy()
for pred in sahi_result.object_prediction_list:
    box = pred.bbox
    x1,y1,x2,y2 = int(box.minx),int(box.miny),int(box.maxx),int(box.maxy)
    cv2.rectangle(sahi_vis,(x1,y1),(x2,y2),(255,100,0),2)

fig, axes = plt.subplots(1,3,figsize=(18,5))
axes[0].imshow(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
axes[0].set_title('Original', fontweight='bold'); axes[0].axis('off')

axes[1].imshow(cv2.cvtColor(std_vis, cv2.COLOR_BGR2RGB))
axes[1].set_title(f'Standard ({len(std_result.boxes)} detections)', fontweight='bold'); axes[1].axis('off')

axes[2].imshow(cv2.cvtColor(sahi_vis, cv2.COLOR_BGR2RGB))
axes[2].set_title(f'SAHI 320×320 tiles ({len(sahi_result.object_prediction_list)} detections)', fontweight='bold'); axes[2].axis('off')

plt.suptitle('SAHI Sliced Inference for Small Object Detection', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(str(RESULTS_DIR / 'sahi_comparison.png'), dpi=150, bbox_inches='tight')
plt.show()